In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

In [2]:
df = pd.read_csv("cleaned_reviews.csv")

# Remove neutral for better model accuracy
df_model = df[df['sentiment'] != 'Neutral'].copy()

print("Total rows for model:", len(df_model))
print("Sentiment distribution:")
print(df_model['sentiment'].value_counts())

# Features and target
X = df_model['clean_text']
y = df_model['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("\nTraining size:", len(X_train))
print("Testing size:", len(X_test))

# Convert text to numbers using TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF matrix shape:", X_train_tfidf.shape)

# Train Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = nb_model.predict(X_test_tfidf)
print("\n Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save model and vectorizer
pickle.dump(nb_model, open('sentiment_model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))
print("\n Model saved!")


Total rows for model: 460379
Sentiment distribution:
sentiment
Positive    340595
Negative    119784
Name: count, dtype: int64

Training size: 368303
Testing size: 92076

TF-IDF matrix shape: (368303, 10000)

 Accuracy: 0.9408097658456058

Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.92      0.89     23810
    Positive       0.97      0.95      0.96     68266

    accuracy                           0.94     92076
   macro avg       0.92      0.93      0.92     92076
weighted avg       0.94      0.94      0.94     92076


 Model saved!


In [3]:
# Load saved model
model = pickle.load(open('sentiment_model.pkl', 'rb'))
tfidf = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))

# Test with your own sentences
test_reviews = [
    "This app is amazing! Super fast delivery!",
    "Worst experience ever. Order cancelled without reason.",
    "Great service and excellent support team",
    "Pathetic app. Refund not received for 2 weeks!",
    "Zomato is the best food delivery app in India",
    "Swiggy cancelled my order at last minute terrible"
]

print("Testing Sentiment Model\n")
for review in test_reviews:
    cleaned = review.lower()
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    print(f"Review: {review[:50]}")
    print(f"Prediction: {prediction}\n")

Testing Sentiment Model

Review: This app is amazing! Super fast delivery!
Prediction: Positive

Review: Worst experience ever. Order cancelled without rea
Prediction: Negative

Review: Great service and excellent support team
Prediction: Positive

Review: Pathetic app. Refund not received for 2 weeks!
Prediction: Negative

Review: Zomato is the best food delivery app in India
Prediction: Positive

Review: Swiggy cancelled my order at last minute terrible
Prediction: Negative

